<a href="https://colab.research.google.com/github/beingAnujChaudhary/Marketing-A-B-Testing-Hypothesis-Testing-Framework/blob/main/notebooks/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q pyyaml statsmodels plotly
    !git clone https://github.com/beingAnujChaudhary/Marketing-A-B-Testing-Hypothesis-Testing-Framework.git
    %cd Marketing-A-B-Testing-Hypothesis-Testing-Framework

from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

In [4]:
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'

from src.config import get_config
from src.data_validation import run_validation
from src.statistical_tests import z_test_proportions, segment_analysis
from src.decision_framework import generate_recommendation
from src.power_analysis import calculate_achieved_power
from src.reporting import save_results_json, plot_conversion_rates, plot_cumulative_lift

In [5]:
config = get_config()
df = pd.read_csv('data/processed/clean_ab_data.csv')

print(f"📊 Dataset: {len(df):,} users")
validation = run_validation(df, config)
if not validation['passed']:
    for warning in validation['warnings']:
        print(f"  • {warning}")
else:
    print("✅ Data validation passed")

📊 Dataset: 5,000 users
✅ Data validation passed


In [6]:
stats = z_test_proportions(df)

print(f"🎯 Conversion Rates:")
print(f"  • Variant A (Control): {stats['conversion_rates']['A']:.2%}")
print(f"  • Variant B (Treatment): {stats['conversion_rates']['B']:.2%}")
print(f"\n📈 Relative Lift: {stats['lift_percent']:+.2f}%")
print(f"🔍 P-value: {stats['p_value']:.4f}")
print(f"🎯 95% CI: [{stats['ci_95'][0]:+.2f}%, {stats['ci_95'][1]:+.2f}%]")
print(f"📏 Effect Size (Cohen's h): {stats['effect_size_cohens_h']}")

🎯 Conversion Rates:
  • Variant A (Control): 11.63%
  • Variant B (Treatment): 14.19%

📈 Relative Lift: +22.08%
🔍 P-value: 0.0069
🎯 95% CI: [+6.12%, +38.05%]
📏 Effect Size (Cohen's h): -0.077


In [7]:
n_per_group = len(df[df['variant'] == 'A'])
power = calculate_achieved_power(
    baseline_rate=config['statistics']['baseline_conversion'],
    observed_lift=stats['lift_percent'] / 100,
    n_per_group=n_per_group
)
print(f"⚡ Achieved Power: {power:.0%}")

unsub_rates = df.groupby('variant')['unsubscribed'].mean()
threshold = config['data']['guardrail_unsubscribe_rate']
guardrail_breached = unsub_rates['B'] > threshold * 1.5
print(f"{'🚨 GUARDRAIL BREACHED' if guardrail_breached else '✅ Guardrail OK'}")

⚡ Achieved Power: 77%
✅ Guardrail OK


In [8]:
recommendation = generate_recommendation(
    stats_result=stats,
    guardrail_breached=guardrail_breached,
    achieved_power=power
)
print(f"🎯 Recommendation: {recommendation['recommendation']}")
print(f"🔐 Confidence: {recommendation['confidence']}")
print(f"📝 Reasoning: {recommendation['reasoning']}")

🎯 Recommendation: ✅ SCALE VARIANT B
🔐 Confidence: HIGH
📝 Reasoning: Statistically significant lift exceeding the business threshold with the entire CI above zero.


In [9]:
fig1 = plot_conversion_rates(df, output_path=None)
fig1.show()

fig2 = plot_cumulative_lift(df, output_path=None)
fig2.show()

In [10]:
segment_results = segment_analysis(df)
if not segment_results.empty:
    display_cols = ['segment', 'lift_percent', 'p_value', 'p_value_corrected', 'significant_corrected']
    print(segment_results[display_cols].to_markdown(index=False))

| segment         |   lift_percent |   p_value |   p_value_corrected | significant_corrected   |
|:----------------|---------------:|----------:|--------------------:|:------------------------|
| New_SME         |        23.8775 | 0.0369595 |            0.110878 | False                   |
| Established_SME |        16.3709 | 0.230528  |            0.349489 | False                   |
| Enterprise      |        30.6616 | 0.174745  |            0.349489 | False                   |


In [11]:
save_results_json(stats, recommendation)
print("✅ Results saved to output/results/summary.json")

if 'google.colab' in sys.modules:
    plot_conversion_rates(df, output_path='/content/drive/MyDrive/ab-test-results/conversion_plot.html')
    print("📊 Charts saved to Google Drive")

✅ Results saved to output/results/summary.json
📊 Charts saved to Google Drive


In [20]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/ab-test-results/conversion_plot.html"
)

from google.colab import files

files.download("output/results/summary.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>